# MOS amplifier building blocks

Schematics and SPICE subcircuits for four single-stage topologies, each available in an
NMOS or PMOS version:

| Topology | Function | Terminals |
|---|---|---|
| Common source | `common_source` | `in`, `out` |
| Source follower | `source_follower` | `in`, `out` |
| Common gate | `common_gate` | `in`, `out` |
| Differential pair | `differential_pair` | `inp`, `inn`, `outp`, `outn` |

Every circuit is biased by an **ideal current source**, and the signal is **capacitively
coupled** in and out, with a load resistor `RL` on each output and a gate-bias resistor
`RG` on each driven gate. The supplies are the global nodes **`vdd!`** and **`vss!`**:
the drawings use `vdd`/`vss`, and the subcircuits rename them. A netlist that uses these
subcircuits must declare them with `.global vdd! vss!` and drive them; the testbench at the
end does that with ±5 V.

Each function takes `type=:nmos` (default) or `:pmos`, and `model`, which defaults to
`ald1105n` or `ald1105p` to match. It returns `(schematic, subckt)`: an annotated drawing
and a Microelectronics.jl `Subcircuit`.

In [ ]:
using SchematicSVG, Microelectronics

## Shared pieces

All circuits use the same conventions, so the helpers are small: check the device type,
pick its default model, describe the transistor (model and size), and turn an annotated
drawing into a subcircuit with the global supplies.

In [ ]:
const GLOBAL_SUPPLIES = Dict("vdd" => "vdd!", "vss" => "vss!")

function checktype(type)
    type in (:nmos, :pmos) || throw(ArgumentError("type must be :nmos or :pmos, got $(repr(type))"))
    type
end

default_model(type) = checktype(type) == :nmos ? "ald1105n" : "ald1105p"

# Instance name of transistor k in the drawings: N1, N2 for NMOS, P1, P2 for PMOS.
fetname(type, k=1) = string(type == :nmos ? "N" : "P", k)

# Model and size of a transistor, as Microelectronics' NMOS/PMOS keywords.
fet(model, W, L) = (; model, W, L)

# Draw the circuit with its values, then build the subcircuit from the same drawing.
function build(drawing, values, name, terminals)
    schematic = ascii_schematic(drawing; elements=values)
    subckt = to_subcircuit(schematic; name, terminals, rename=GLOBAL_SUPPLIES)
    (; schematic, subckt)
end

## Common source

The source is biased by `I1` and bypassed to ground by `CB`, so at signal frequencies the
gain is $A_v \approx -g_m (R_D \parallel R_L \parallel r_o)$. The gate sits at 0 V through
`RG`. In the PMOS version the stack is upside down: `I1` feeds the source from `vdd` and
`RD` returns the drain to `vss`.

In [ ]:
const CS_DRAWING = Dict(
:nmos => """
                         vdd
                          |
                          RD
                          |
                          +-@d-CC2----+--out
                          |           |
  in----CC1--@g----+-----|N1          RL
                   |      |           |
                   RG     +-@s-+     gnd
                   |      |    |
                  gnd     I1   CB
                          |    |
                         vss  gnd
""",
:pmos => """
                         vdd
                          |
                          I1
                          |
                          +-@s-+
                          |    |
  in----CC1--@g----+-----|P1   CB
                   |      |    |
                   RG     |   gnd
                   |      |
                  gnd     +-@d-CC2----+--out
                          |           |
                          RD          RL
                          |           |
                         vss         gnd
""")

"""
    common_source(; type=:nmos, model, name, W=20μm, L=10μm, Ibias=100μA, RD=20kΩ,
                  RG=1MΩ, RL=100kΩ, CC=10μF, CB=100μF)

Capacitively coupled common-source amplifier with an ideal bias current source and a
bypassed source. Terminals: `in`, `out`.
"""
function common_source(; type=:nmos, model=default_model(type), name="cs_$(type)",
                       W=20u"μm", L=10u"μm", Ibias=100u"μA", RD=20u"kΩ", RG=1u"MΩ",
                       RL=100u"kΩ", CC=10u"μF", CB=100u"μF")
    checktype(type)
    values = Dict{String,Any}(fetname(type) => fet(model, W, L), "I1" => Ibias, "RD" => RD,
                              "RG" => RG, "RL" => RL, "CC1" => CC, "CC2" => CC, "CB" => CB)
    build(CS_DRAWING[type], values, name, ["in", "out"])
end

In [ ]:
cs = common_source()
cs.schematic

In [ ]:
print(cs.subckt)

In [ ]:
common_source(type=:pmos).schematic

## Source follower

The drain goes straight to a supply and the output is taken from the source, so the gain
is $A_v \approx g_m R_L / (1 + g_m R_L)$, just under 1. `I1` sets the bias current.

In [ ]:
const SF_DRAWING = Dict(
:nmos => """
                         vdd
                          |
                          |
                          |
  in----CC1--@g----+-----|N1
                   |      |
                   RG     +-@s-CC2----+--out
                   |      |           |
                  gnd     I1          RL
                          |           |
                         vss         gnd
""",
:pmos => """
                         vdd
                          |
                          I1
                          |
                          +-@s-CC2----+--out
                          |           |
  in----CC1--@g----+-----|P1          RL
                   |      |           |
                   RG     |          gnd
                   |      |
                  gnd    vss
""")

"""
    source_follower(; type=:nmos, model, name, W=20μm, L=10μm, Ibias=100μA, RG=1MΩ,
                    RL=100kΩ, CC=10μF)

Capacitively coupled source follower with an ideal bias current source.
Terminals: `in`, `out`.
"""
function source_follower(; type=:nmos, model=default_model(type), name="sf_$(type)",
                         W=20u"μm", L=10u"μm", Ibias=100u"μA", RG=1u"MΩ", RL=100u"kΩ",
                         CC=10u"μF")
    checktype(type)
    values = Dict{String,Any}(fetname(type) => fet(model, W, L), "I1" => Ibias, "RG" => RG,
                              "RL" => RL, "CC1" => CC, "CC2" => CC)
    build(SF_DRAWING[type], values, name, ["in", "out"])
end

In [ ]:
sf = source_follower()
sf.schematic

In [ ]:
print(sf.subckt)

In [ ]:
source_follower(type=:pmos).schematic

## Common gate

The gate is grounded and the signal enters at the source, which `I1` biases. The gain is
$A_v \approx +g_m (R_D \parallel R_L)$ when the input is driven by a low-impedance source,
and the input resistance is about $1/g_m$.

In [ ]:
const CG_DRAWING = Dict(
:nmos => """
                         vdd
                          |
                          RD
                          |
                          +-@d-CC2----+--out
                          |           |
                   gnd---|N1          RL
                          |           |
  in----CC1--------@s-----+          gnd
                          |
                          I1
                          |
                         vss
""",
:pmos => """
                         vdd
                          |
                          I1
                          |
  in----CC1--------@s-----+
                          |
                   gnd---|P1
                          |
                          +-@d-CC2----+--out
                          |           |
                          RD          RL
                          |           |
                         vss         gnd
""")

"""
    common_gate(; type=:nmos, model, name, W=20μm, L=10μm, Ibias=100μA, RD=20kΩ,
                RL=100kΩ, CC=10μF)

Capacitively coupled common-gate amplifier with an ideal bias current source.
Terminals: `in`, `out`.
"""
function common_gate(; type=:nmos, model=default_model(type), name="cg_$(type)",
                     W=20u"μm", L=10u"μm", Ibias=100u"μA", RD=20u"kΩ", RL=100u"kΩ",
                     CC=10u"μF")
    checktype(type)
    values = Dict{String,Any}(fetname(type) => fet(model, W, L), "I1" => Ibias, "RD" => RD,
                              "RL" => RL, "CC1" => CC, "CC2" => CC)
    build(CG_DRAWING[type], values, name, ["in", "out"])
end

In [ ]:
cg = common_gate()
cg.schematic

In [ ]:
print(cg.subckt)

In [ ]:
common_gate(type=:pmos).schematic

## Differential pair

`I1` is the tail current, shared by the two transistors (`Ibias/2` each when balanced).
Each gate has its own coupling capacitor and bias resistor, and each drain its own output
coupling and load. `outn` is the drain on the `inp` side, so it falls when `inp` rises:
the differential gain is $(v_{outp} - v_{outn})/(v_{inp} - v_{inn}) \approx g_m (R_D
\parallel R_L)$, half of that from each output.

In [ ]:
const DIFF_DRAWING = Dict(
:nmos => """
                                 vdd           vdd
                                  |             |
    outn                          RD1           RD2                       outp
    |                             |             |                            |
    +----CC3------------@d1-------+             +---@d2------------CC4-------+
    |                             |             |                            |
    RL1  inp--CC1--@g1-----+-----|N1            N2|----+-@g2---CC2--inn      RL2
    |                      |      |             |      |                     |
   gnd                     RG1    +-@tail+------+      RG2                  gnd
                           |             |             |
                          gnd            I1           gnd
                                         |
                                        vss
""",
:pmos => """
                                        vdd
                                         |
                                         I1
                                         |
                                  +-@tail+------+
                                  |             |
         inp--CC1--@g1-----+-----|P1            P2|----+-@g2---CC2--inn
                           |      |             |      |
                           RG1    |             |      RG2
                           |      |             |      |
    outn                  gnd     |             |     gnd                 outp
    |                             |             |                            |
    +----CC3------------@d1-------+             +---@d2------------CC4-------+
    |                             |             |                            |
    RL1                           RD1           RD2                          RL2
    |                             |             |                            |
   gnd                           vss           vss                          gnd
""")

"""
    differential_pair(; type=:nmos, model, name, W=20μm, L=10μm, Ibias=200μA, RD=20kΩ,
                      RG=1MΩ, RL=100kΩ, CC=10μF)

Capacitively coupled differential pair with an ideal tail current source `Ibias`.
Terminals: `inp`, `inn`, `outp`, `outn`.
"""
function differential_pair(; type=:nmos, model=default_model(type), name="diff_$(type)",
                           W=20u"μm", L=10u"μm", Ibias=200u"μA", RD=20u"kΩ", RG=1u"MΩ",
                           RL=100u"kΩ", CC=10u"μF")
    checktype(type)
    m = fet(model, W, L)
    values = Dict{String,Any}(fetname(type, 1) => m, fetname(type, 2) => m, "I1" => Ibias,
                              "RD1" => RD, "RD2" => RD, "RG1" => RG, "RG2" => RG,
                              "RL1" => RL, "RL2" => RL,
                              "CC1" => CC, "CC2" => CC, "CC3" => CC, "CC4" => CC)
    build(DIFF_DRAWING[type], values, name, ["inp", "inn", "outp", "outn"])
end

In [ ]:
dp = differential_pair()
dp.schematic

In [ ]:
print(dp.subckt)

In [ ]:
differential_pair(type=:pmos).schematic

## Testbench

`testbench` places one subcircuit between ±5 V global supplies, drives its input with a
10 mV, 1 kHz sine (for the differential pair, `inp` is driven and `inn` is grounded), finds
the operating point, and measures the peak-to-peak gain over a transient run. The bias
voltages are those of the tagged nodes inside the subcircuit (`g`, `s`, `d`, or `g1`,
`tail`, `d1`, `d2`), which ngspice names `x1.d` and so on.

In [ ]:
const PROBES = Dict("cs" => ["g", "s", "d"], "sf" => ["g", "s"], "cg" => ["s", "d"],
                    "diff" => ["g1", "tail", "d1", "d2"])

nodevoltage(n) = only(Microelectronics.NgHerb.getrealvec(lowercase(n)))

function testbench(sub::Subcircuit; vdd=5u"V", vss=-5u"V", amplitude=10u"mV", frequency=1u"kHz")
    diff = length(sub.terminals) == 4
    inp, out = diff ? ("inp", "outp") : ("in", "out")
    sources = [Vdc("VDD", "vdd!", 0, vdd), Vdc("VSS", "vss!", 0, vss),
               Vsin("Vsig", inp, 0; amplitude, frequency)]
    diff && push!(sources, Vdc("Vn", "inn", 0, 0u"V"))
    components = [Subckt("X1", sub.terminals, sub.name)]
    redirect_stdout(devnull) do
        build_netlist(; title="testbench for $(sub.name)", parameters=".global vdd! vss!",
                      models=string(sub), sources, components)
        simulate(:op)
    end
    bias = Dict(n => nodevoltage("x1.$n") * u"V" for n in get(PROBES, first(split(sub.name, '_')), String[]))
    redirect_stdout(devnull) do
        simulate(:tran; tstep=1/(100frequency), tmax=5/frequency)
    end
    gain = measure(:Av; in=inp, out=out)
    (; bias, gain)
end

In [ ]:
testbench(cs.subckt)

The waveforms of the last run:

In [ ]:
t = Microelectronics.NgHerb.getrealvec("time") .* 1e3            # ms
vin, vout = Microelectronics.NgHerb.getrealvec("in"), Microelectronics.NgHerb.getrealvec("out")
plot(t, [vin vout] .* 1e3; labels=["in" "out"], xlabel="Time [ms]", ylabel="Voltage [mV]")

## All eight circuits

The table runs every topology in both polarities. In a check with ngspice and the
ALD1105 models at the default values, the drains sat at ±3 V and the gains were about
5.1 (NMOS) and 3.4 (PMOS) for the common-source and common-gate stages, 0.96 and 0.95 for
the followers, and 2.5 and 1.7 from each output of the differential pairs (the PMOS
devices have the lower $k_p$).

In [ ]:
for f in (common_source, source_follower, common_gate, differential_pair), type in (:nmos, :pmos)
    sub = f(; type).subckt
    r = testbench(sub)
    biases = join(["$k = $(round(u"V", v; digits=2))" for (k, v) in sort(collect(r.bias))], ", ")
    println(rpad(sub.name, 11), "  |Av| = ", round(r.gain; digits=2), "    ", biases)
end

## Using a subcircuit in your own netlist

Include the definition with `models=string(sub)` (or several, joined), declare the global
supplies, drive them, and instantiate the subcircuit with `Subckt`:

```julia
amp = common_source(type=:pmos, RD=15u"kΩ").subckt
build_netlist(; parameters=".global vdd! vss!", models=string(amp),
              sources=[Vdc("VDD", "vdd!", 0, 5u"V"), Vdc("VSS", "vss!", 0, -5u"V"),
                       Vsin("Vs", "a", 0; amplitude=10u"mV", frequency=1u"kHz")],
              components=[Subckt("X1", ["a", "b"], amp.name), Resistor("RX", "b", 0, 10u"kΩ")])
```